# Thesis numbers not covered by the other notebooks

In [1]:
import os, re, json, glob, csv, sys
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
csv.field_size_limit(sys.maxsize)
sys.path.insert(0, "audit")
import error_categories as ec
import opencode_failures as of

FINAL = Path("../results"); DATA = Path("../data"); RUNS = Path("../runs")
CACHE = FINAL / "_analysis_cache"; TABLES = FINAL / "repair_loop_tables"
RES = Path("result/final"); INTENT = Path("ablation_selection/intent_results")
CFN_META = Path("../benchmark/cloudformation/final_benchmark_real_aws_with_prompts.csv")
TF_META = Path("../benchmark/terraform/final_benchmark_real_aws_with_prompts.csv")
TF_REGISTRY = Path("../tf_registry_docs.json")
BUDGET = 15
pd.set_option("display.width", 220, "display.max_columns", 40)

def extract(name, build, sources):
    p = CACHE / f"thesis_{name}.csv"
    if all(Path(s).exists() for s in sources):
        d = build(); d.to_csv(p, index=False)
        return d
    return pd.read_csv(p)

COLS = ["row_number", "run_id", "validity", "final_validation_passed", "iterations_used", "llm_calls_total", "ground_truth_path"]
def load(e):
    st = json.load(open(FINAL / e / "status.json"))
    r = pd.read_csv(FINAL / e / "results_final.csv", engine="python", usecols=lambda c: c in COLS)
    ds = pd.read_csv(DATA / st["dataset"]); dcol = "difficulty" if "difficulty" in ds else "difficulty_level"
    r["row_number"] = r.row_number.astype(int)
    r["level"] = r.row_number.map(ds.assign(row_number=ds.row_number.astype(int)).drop_duplicates("row_number").set_index("row_number")[dcol])
    r["pass"] = (r.validity == "valid_pass") & (r.iterations_used <= BUDGET)
    return r

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

## 1. Benchmark sources and hand repairs

In [2]:
cm = pd.read_csv(CFN_META); tm = pd.read_csv(TF_META, low_memory=False)
pd.Series({"TF scenarios from TerraDS": int((tm.source == "Zenodo/TerraDS").sum()),
           "CFN scenarios from DPIaC-Eval (Tianyi2/IaCGen)": int((cm.source_slug == "Tianyi2/IaCGen").sum()),
           "CFN scenarios repaired by hand": int((cm.source_category == "manual-fix").sum()),
           "TF scenarios repaired by hand": int((tm.source_category == "manual-fix").sum())}).to_frame("scenarios")

,scenarios
TF scenarios from TerraDS,221
CFN scenarios from DPIaC-Eval (Tianyi2/IaCGen),7
CFN scenarios repaired by hand,52
TF scenarios repaired by hand,114


## 2. Medium and low findings among Trivy-clean candidates

In [3]:
def build_clean_candidates():
    C = Path("cfn_benchmark/dataset"); T = Path("iac_benchmark/dataset")
    aws = set(pd.read_csv(C / "df_aws_cache.csv", usecols=["content_hash"]).content_hash)
    lint = pd.read_csv(C / "lint_cache.csv", usecols=["content_hash", "lint_pass"])
    lp = set(lint.loc[(lint.lint_pass == True) & lint.content_hash.isin(aws), "content_hash"])
    sec = pd.read_csv(C / "security_cache.csv", usecols=["content_hash", "trivy_critical", "trivy_high", "trivy_medium", "trivy_low", "trivy_error"]).drop_duplicates("content_hash", keep="last")
    sec = sec[sec.content_hash.isin(lp) & (sec.trivy_error.fillna("") == "")]
    tv = pd.read_csv(T / "trivy_scanned_batch.csv", usecols=["scenario_id", "trivy_critical", "trivy_high", "trivy_medium", "trivy_low"], low_memory=False)
    out = []
    for lang, d, key in (("CFN", sec, "content_hash"), ("TF", tv, "scenario_id")):
        d = d[(d.trivy_critical == 0) & (d.trivy_high == 0)]
        out.append(pd.DataFrame({"lang": lang, "id": d[key], "trivy_medium": d.trivy_medium.fillna(0).astype(int), "trivy_low": d.trivy_low.fillna(0).astype(int)}))
    return pd.concat(out, ignore_index=True)
cand = extract("clean_candidates", build_clean_candidates, ["cfn_benchmark/dataset/security_cache.csv", "iac_benchmark/dataset/trivy_scanned_batch.csv"])
cand.assign(medium_or_low=(cand.trivy_medium > 0) | (cand.trivy_low > 0)).groupby("lang").agg(
    clean_candidates=("id", "size"), with_medium_or_low=("medium_or_low", "sum"), share_pct=("medium_or_low", lambda s: round(100 * s.mean(), 1)))

,clean_candidates,with_medium_or_low,share_pct
lang,,,
CFN,9351,3293,35.2
TF,36963,12649,34.2


## 3. Size and prompt length by level, and the example scenarios

In [4]:
ce = pd.read_csv(DATA / "cfn_eval_benchmark_real_aws.csv"); te = pd.read_csv(DATA / "tf_eval_benchmark_real_aws.csv")
lv = []
for lang, ev, meta in (("CFN", ce, cm), ("TF", te, tm)):
    d = ev.assign(row_number=ev.row_number.astype(int)).merge(meta[["row_number", "loc", "n_resources"]], on="row_number")
    d["words"] = d.prompt.str.split().str.len(); d["lang"] = lang; lv.append(d)
lv = pd.concat(lv, ignore_index=True)
med = lv.groupby(["lang", "difficulty"]).agg(LoC=("loc", "median"), Res=("n_resources", "median"), Prompt=("words", "median"))
display(med.unstack(0))
print("level-5 / level-1 median prompt words:", {l: round(med.loc[(l, 5), "Prompt"] / med.loc[(l, 1), "Prompt"], 2) for l in ("CFN", "TF")})
lv[((lv.lang == "CFN") & lv.row_number.isin([9, 165])) | ((lv.lang == "TF") & (lv.row_number == 43))][["lang", "row_number", "difficulty", "loc", "n_resources"]]

LoC          Res       Prompt       
lang          CFN     TF   CFN    TF    CFN     TF
difficulty                                        
1            43.0   54.0   2.0   3.0   54.5   75.5
2           121.0  110.5   4.0   4.0  107.5   64.0
3           217.5  209.5   7.0   7.0  139.0  162.0
4           320.5  330.5  12.0  14.0  151.5  188.0
5           514.0  540.0  20.0  18.0  191.0  226.0

level-5 / level-1 median prompt words: {'CFN': 3.5, 'TF': 2.99}


,lang,row_number,difficulty,loc,n_resources
9,CFN,9,1,35.0,2.0
155,CFN,165,5,426.0,12.0
288,TF,43,2,126.0,6.0


## 4. Services and resource types

In [5]:
reg = extract("tf_registry_subcategories", lambda: pd.DataFrame(
    [(k, v["subcategory"]) for k, v in json.loads(TF_REGISTRY.read_text()).items() if v.get("subcategory")], columns=["type", "subcategory"]), [TF_REGISTRY])
TF_TYPE_TO_SERVICE = dict(zip(reg.type, reg.subcategory))
SERVICE_ALIAS = {"Logs": "CloudWatch Logs", "Events": "EventBridge", "Cognito IDP": "Cognito", "Cognito Identity": "Cognito",
                 "ApiGateway": "API Gateway", "ApiGatewayV2": "API Gateway V2", "StepFunctions": "SFN", "SecretsManager": "Secrets Manager",
                 "Route53": "Route 53", "ElasticLoadBalancingV2": "ELB", "ELB Classic": "ELB"}
SAM_TO_SERVICE = {"Function": "Lambda", "LayerVersion": "Lambda", "Api": "API Gateway", "HttpApi": "API Gateway V2",
                  "SimpleTable": "DynamoDB", "StateMachine": "SFN"}
RESOURCE_ALIAS = {"AWS::Logs::LogGroup": "aws_cloudwatch_log_group", "AWS::EC2::SubnetRouteTableAssociation": "aws_route_table_association",
                  "AWS::EC2::VPCGatewayAttachment": "aws_internet_gateway", "AWS::IAM::ManagedPolicy": "aws_iam_policy",
                  "AWS::IAM::Policy": "aws_iam_role_policy", "AWS::IAM::RolePolicy": "aws_iam_role_policy"}
EXCLUDED_NAMESPACES = {"CDK", "CloudFormation"}
RESOURCE_LABELS = {"aws_iam_role": "IAM role", "aws_vpc": "VPC", "aws_iam_policy": "Managed policy", "aws_iam_role_policy": "Inline policy",
                   "aws_subnet": "Subnet", "aws_route_table": "Route table", "aws_internet_gateway": "Internet gateway", "aws_kms_key": "KMS key",
                   "aws_route_table_association": "Route table assoc.", "aws_security_group": "Security group",
                   "aws_iam_role_policy_attachment": "Role policy attach.", "aws_eip": "Elastic IP"}

def _snake(name): return re.sub(r"(?<=[a-z0-9])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])", "_", name).lower()
def _short_service(sub):
    label = re.sub(r"\s*\(.*\)$", "", sub).strip()
    return SERVICE_ALIAS.get(label, label)
def cfn_to_tf_type(t):
    if t in RESOURCE_ALIAS: return RESOURCE_ALIAS[t]
    m = re.match(r"AWS::(\w+)::(\w+)$", t)
    if not m: return t
    ns, rt = m.groups()
    for cand_t in (f"aws_{_snake(ns)}_{_snake(rt)}", f"aws_{_snake(rt)}"):
        if cand_t in TF_TYPE_TO_SERVICE: return cand_t
    return t
def cfn_service(t):
    m = re.match(r"AWS::(\w+)::(\w+)$", t)
    if not m: return None
    ns, rt = m.groups()
    if ns in EXCLUDED_NAMESPACES: return None
    if ns == "Serverless": return SAM_TO_SERVICE.get(rt, "Lambda")
    tt = cfn_to_tf_type(t)
    if tt in TF_TYPE_TO_SERVICE: return _short_service(TF_TYPE_TO_SERVICE[tt])
    if ns == "EC2": return "VPC"
    return SERVICE_ALIAS.get(ns, ns)
def tf_resource_types(folder):
    types = set()
    for f in Path(folder).glob("*.tf"):
        types.update(re.findall(r'resource\s+"(aws_[a-zA-Z0-9_]+)"', f.read_text(encoding="utf-8", errors="ignore")))
    return types

cfn_sets = [{t for t in str(s).split(" | ") if t} for s in cm.sort_values("row_number").resource_types.fillna("")]
tf_sets = [tf_resource_types(p) for p in te.sort_values("row_number").ground_truth_path]
cfn_services = Counter(s for ts in cfn_sets for s in {cfn_service(t) for t in ts} - {None})
tf_services = Counter(s for ts in tf_sets for s in {_short_service(TF_TYPE_TO_SERVICE[t]) if t in TF_TYPE_TO_SERVICE else t for t in ts})
cfn_types = Counter(t for ts in cfn_sets for t in {cfn_to_tf_type(x) for x in ts})
tf_types = Counter(t for ts in tf_sets for t in ts)
n_cfn, n_tf = len(cfn_sets), len(tf_sets)
display(pd.Series({"CFN services": len(cfn_services), "CFN resource types": len({t for s in cfn_sets for t in s}),
                   "TF services": len(tf_services), "TF resource types": len(tf_types),
                   "services in both": len(set(cfn_services) & set(tf_services)), "services in either": len(set(cfn_services) | set(tf_services)),
                   "CFN templates with IAM %": round(100 * cfn_services["IAM"] / n_cfn, 1),
                   "TF templates with IAM %": round(100 * tf_services["IAM"] / n_tf, 1)}).to_frame("value"))
rank = lambda c, t: lambda k: (-(c[k] + t[k]), -max(c[k] / n_cfn, t[k] / n_tf), k)
top_s = sorted(set(cfn_services) | set(tf_services), key=rank(cfn_services, tf_services))[:12]
top_t = sorted(set(cfn_types) | set(tf_types), key=rank(cfn_types, tf_types))[:12]
svc = pd.DataFrame({"service": top_s, "cfn_pct": [100 * cfn_services[s] / n_cfn for s in top_s], "tf_pct": [100 * tf_services[s] / n_tf for s in top_s]})
typ = pd.DataFrame({"label": [RESOURCE_LABELS.get(t, t) for t in top_t], "cfn_pct": [100 * cfn_types[t] / n_cfn for t in top_t], "tf_pct": [100 * tf_types[t] / n_tf for t in top_t]})
display(svc.round(1)); display(typ.round(1))
pgf = lambda v: " ".join(f"({x:.1f},{i})" for i, x in enumerate(v))
for name, d, col in (("services", svc, "service"), ("resource types", typ, "label")):
    print(f"{name}: yticklabels={{{', '.join(d[col])}}}\n  CFN: {pgf(d.cfn_pct)}\n  TF : {pgf(d.tf_pct)}")

,value
CFN services,77.0
CFN resource types,252.0
TF services,57.0
TF resource types,245.0
services in both,36.0
services in either,98.0
CFN templates with IAM %,58.8
TF templates with IAM %,43.2


,service,cfn_pct,tf_pct
0,IAM,58.8,43.2
1,VPC,18.0,34.0
2,KMS,10.8,22.8
3,EC2,9.2,17.2
4,CloudWatch Logs,16.4,5.2
5,S3,4.4,16.0
6,DynamoDB,12.0,4.0
7,Lambda,14.0,0.4
8,Cognito,10.8,3.6
9,SSM,9.6,3.2


,label,cfn_pct,tf_pct
0,IAM role,56.4,35.6
1,VPC,15.6,28.8
2,Subnet,14.0,25.6
3,Internet gateway,11.2,22.8
4,Route table,12.4,21.6
5,KMS key,10.8,22.8
6,Route table assoc.,12.4,20.8
7,Managed policy,6.0,21.2
8,Security group,7.2,18.4
9,Role policy attach.,0.0,24.8


services: yticklabels={IAM, VPC, KMS, EC2, CloudWatch Logs, S3, DynamoDB, Lambda, Cognito, SSM, EventBridge, CloudWatch}
  CFN: (58.8,0) (18.0,1) (10.8,2) (9.2,3) (16.4,4) (4.4,5) (12.0,6) (14.0,7) (10.8,8) (9.6,9) (6.8,10) (6.8,11)
  TF : (43.2,0) (34.0,1) (22.8,2) (17.2,3) (5.2,4) (16.0,5) (4.0,6) (0.4,7) (3.6,8) (3.2,9) (2.0,10) (1.2,11)
resource types: yticklabels={IAM role, VPC, Subnet, Internet gateway, Route table, KMS key, Route table assoc., Managed policy, Security group, Role policy attach., Elastic IP, Inline policy}
  CFN: (56.4,0) (15.6,1) (14.0,2) (11.2,3) (12.4,4) (10.8,5) (12.4,6) (6.0,7) (7.2,8) (0.0,9) (9.2,10) (13.2,11)
  TF : (35.6,0) (28.8,1) (25.6,2) (22.8,3) (21.6,4) (22.8,5) (20.8,6) (21.2,7) (18.4,8) (24.8,9) (15.2,10) (10.8,11)


## 5. Prior benchmarks: security findings over the references Trivy can parse

In [6]:
def build_prior():
    O = Path("output/benchmark_security_audit"); D = Path("output/dpiac_security_audit")
    u = pd.read_csv(O / "benchmark_combined_annotations.csv", usecols=["scenario_id", "source", "language"])
    tv = pd.read_csv(O / "trivy_cache.csv", usecols=["scenario_id", "trivy_error", "severity_critical", "severity_high"]).drop_duplicates("scenario_id", keep="last")
    a = u.merge(tv, on="scenario_id", how="left")
    a["benchmark"] = a.source.map({"AmazonScience/Multi-IaC-Eval": "Multi-IaC-Eval", "autoiac-project/iac-eval": "IaC-Eval"}) + " " + a.language.map({"cloudformation": "CFN", "terraform": "TF"})
    d = pd.read_csv(D / "dpiac_gt_trivy_cache.csv", usecols=["row_number", "trivy_error", "severity_critical", "severity_high"])
    d = d.rename(columns={"row_number": "scenario_id"}).assign(benchmark="DPIaC-Eval CFN")
    return pd.concat([a[["benchmark", "scenario_id", "trivy_error", "severity_critical", "severity_high"]], d], ignore_index=True)
prior = extract("prior_benchmark_trivy", build_prior, ["output/benchmark_security_audit/trivy_cache.csv", "output/dpiac_security_audit/dpiac_gt_trivy_cache.csv"])
prior["parsed"] = prior.trivy_error.fillna("") != "not_scanned_by_trivy"
prior["no_crit_high"] = (prior.severity_critical == 0) & (prior.severity_high == 0)
p = prior[prior.parsed]
p.groupby("benchmark").agg(references=("scenario_id", "size"), no_crit_high=("no_crit_high", "sum"),
                           no_crit_high_pct=("no_crit_high", lambda s: round(100 * s.mean(), 1)),
                           crit_high_pct=("no_crit_high", lambda s: round(100 * (1 - s.mean()), 1)))

,references,no_crit_high,no_crit_high_pct,crit_high_pct
benchmark,,,,
DPIaC-Eval CFN,146,56,38.4,61.6
IaC-Eval TF,369,159,43.1,56.9
Multi-IaC-Eval CFN,160,48,30.0,70.0
Multi-IaC-Eval TF,221,145,65.6,34.4


## 6. Security knowledge base

In [7]:
sc = pd.DataFrame(json.load(open(DATA / "security_checks.json")).values())
has = lambda c: sc[c].fillna("").astype(str).str.strip().replace("[]", "").ne("")
pd.Series({"checks": len(sc), "with severity": int(has("severity").sum()),
           "with severity and Rego source": int((has("severity") & has("source_code")).sum()),
           "CFN remediation and example": int((has("remediation_cfn") & has("cfn_good_example")).sum()),
           "TF remediation and example": int((has("remediation_tf") & has("tf_good_example")).sum())}).to_frame("checks")

,checks
checks,732
with severity,175
with severity and Rego source,175
CFN remediation and example,112
TF remediation and example,129


## 7. Zero-shot pass rate by level

In [8]:
ZS = [f"{l}_RealAWS_{m}" for l in ("CFN", "TF") for m in ("OneShot_DSV4F", "OneShot_GLM53F", "OneShot_Gemini38F", "ZeroShot_Opus55")]
R = {e: load(e) for e in ZS}
pd.DataFrame({e: {**{"All": 100 * r["pass"].mean()}, **{f"L{k}": 100 * v for k, v in r.groupby("level")["pass"].mean().items()}} for e, r in R.items()}).T.round(1)

,All,L1,L2,L3,L4,L5
CFN_RealAWS_OneShot_DSV4F,38.8,54.0,56.0,36.0,22.0,26.0
CFN_RealAWS_OneShot_GLM53F,27.2,38.0,36.0,24.0,22.0,16.0
CFN_RealAWS_OneShot_Gemini38F,50.4,70.0,60.0,40.0,38.0,44.0
CFN_RealAWS_ZeroShot_Opus55,63.6,78.0,78.0,52.0,56.0,54.0
TF_RealAWS_OneShot_DSV4F,44.4,56.0,46.0,48.0,42.0,30.0
TF_RealAWS_OneShot_GLM53F,29.6,48.0,40.0,30.0,24.0,6.0
TF_RealAWS_OneShot_Gemini38F,38.4,60.0,46.0,38.0,30.0,18.0
TF_RealAWS_ZeroShot_Opus55,71.2,86.0,62.0,82.0,70.0,56.0


## 8. Scenarios that every low-cost model fails

In [9]:
for l in ("CFN", "TF"):
    runs = {m: load(f"{l}_RealAWS_{m}").set_index("row_number")["pass"] for m in ("DSV4F", "GLM53F", "Gemini38F", "Opus55", "OpenCode_DSV4F")}
    P = pd.concat([runs[m].rename(m) for m in ("DSV4F", "GLM53F", "Gemini38F")], axis=1)
    allfail = list(P[~P.any(axis=1)].index)
    print(l, "failed by all three low-cost models:", allfail, "| Claude-Opus-5.5 passes:", runs["Opus55"].loc[allfail].to_dict(),
          "| OpenCode passes:", runs["OpenCode_DSV4F"].loc[allfail].to_dict())

CFN failed by all three low-cost models: [6] | Claude-Opus-5.5 passes: {6: True} | OpenCode passes: {6: False}
TF failed by all three low-cost models: [249] | Claude-Opus-5.5 passes: {249: True} | OpenCode passes: {249: False}


## 9. OpenCode: refused submissions and LLM calls per iteration

In [10]:
def build_refused():
    idx = {os.path.basename(d.rstrip("/")): d for d in glob.glob(str(RUNS / "*" / "*/"))}
    rows = []
    for l in ("CFN", "TF"):
        for rid in load(f"{l}_RealAWS_OpenCode_DSV4F").run_id.astype(str):
            d = idx.get(rid)
            if d and os.path.exists(os.path.join(d, "harness_tool_calls.txt")):
                rows.append((l, rid, sum(k == "SUBMIT_REFUSED" for _, k in of.tool_events(d))))
    return pd.DataFrame(rows, columns=["lang", "run_id", "refused"])
ref = extract("opencode_refused_submissions", build_refused, [RUNS])
out = []
for l in ("CFN", "TF"):
    oc = load(f"{l}_RealAWS_OpenCode_DSV4F").merge(ref[ref.lang == l][["run_id", "refused"]].astype({"run_id": str}), on="run_id", how="left")
    ms = load(f"{l}_RealAWS_DSV4F")
    for name, r in (("OpenCode", oc), ("IaCGen-MAS", ms)):
        r = r.assign(cpi=r.llm_calls_total / r.iterations_used); first = r[r["pass"] & (r.iterations_used == 1)]
        out.append({"lang": l, "system": name, "runs with a refused submission": int((r.get("refused", pd.Series(dtype=float)) > 0).sum()) if name == "OpenCode" else None,
                    "calls/iteration median": r.cpi.median(), "calls/iteration mean": round(r.cpi.mean(), 2), "calls/iteration max": r.cpi.max(),
                    "first-attempt passes": len(first), "their median calls": first.llm_calls_total.median()})
pd.DataFrame(out)

,lang,system,runs with a refused submission,calls/iteration median,calls/iteration mean,calls/iteration max,first-attempt passes,their median calls
0,CFN,OpenCode,8.0,4.954545,5.02,42.000000,79,5.0
1,CFN,IaCGen-MAS,NaN,2.500000,2.47,2.933333,47,2.0
2,TF,OpenCode,3.0,5.000000,4.75,8.000000,134,5.0
3,TF,IaCGen-MAS,NaN,2.333333,2.33,2.928571,100,2.0


## 10. Intent checks per scenario

In [11]:
b = pd.read_csv(INTENT / "baseline.csv")
b.groupby("lang").agg(scenarios=("row_number", "size"), checks_per_scenario=("n_checks", "mean"), core_per_scenario=("n_core", "mean")).assign(
    core_share=b.groupby("lang").n_core.sum() / b.groupby("lang").n_checks.sum()).round(2)

,scenarios,checks_per_scenario,core_per_scenario,core_share
lang,,,,
cfn,50,17.40,9.42,0.54
tf,50,18.56,9.64,0.52


## 11. Ablation failures and the live-AWS run on the same scenarios

In [12]:
CONDS = ("IaCGOD", "NoDenseRAG", "NoGraphRAG", "NoQueryRewrite", "NoRemediator", "NoPlanner", "TrivyAllSeverities")
fails = []
for l in ("CFN", "TF"):
    for c in CONDS:
        r = load(f"{l}_Ablation_{c}")
        fails += [(l, c, n) for n in r.loc[~r["pass"], "row_number"]]
fails = pd.DataFrame(fails, columns=["lang", "condition", "row_number"])
print("failures:", len(fails)); display(fails.groupby(["lang", "row_number"]).condition.nunique().sort_values(ascending=False).to_frame("conditions failed").head(6))
abl = load("CFN_Ablation_IaCGOD"); live = load("CFN_RealAWS_DSV4F")
for key in ("row_number", "ground_truth_path"):
    s = live[live[key].isin(abl[key])]
    print(f"live AWS on the ablation scenarios (by {key}): n={len(s)}, pass {100 * s['pass'].mean():.1f}%, iterations to pass {s[s['pass']].iterations_used.mean():.2f}")
print(f"LocalStack ablation baseline: pass {100 * abl['pass'].mean():.1f}%, iterations to pass {abl[abl['pass']].iterations_used.mean():.2f}")

failures: 16


conditions failed
lang row_number                   
CFN  163                         6
TF   249                         6
CFN  85                          1
     176                         1
TF   143                         1
     174                         1

live AWS on the ablation scenarios (by row_number): n=50, pass 90.0%, iterations to pass 5.56
live AWS on the ablation scenarios (by ground_truth_path): n=50, pass 90.0%, iterations to pass 5.56
LocalStack ablation baseline: pass 98.0%, iterations to pass 3.04


## 12. Planner objectives in the ablation baseline

In [13]:
def build_objectives():
    rows = []
    for l in ("CFN", "TF"):
        for n, rid in zip(*load(f"{l}_Ablation_IaCGOD")[["row_number", "run_id"]].astype({"run_id": str}).values.T):
            j = json.load(open(RUNS / f"Ablation_IaCGOD_{l}_runs" / rid / "final_report.json"))
            o = "\n".join(j.get("objectives") or []); q = j.get("user_request") or ""
            rows.append((l, n, rid, len(j.get("objectives") or []), len(o), len(q), "ImageTagImmutability" in o, "MaxConcurrentRuns" in o))
    return pd.DataFrame(rows, columns=["lang", "row_number", "run_id", "n_objectives", "objective_chars", "request_chars", "ImageTagImmutability", "MaxConcurrentRuns"])
obj = extract("planner_objectives", build_objectives, [RUNS / "Ablation_IaCGOD_CFN_runs", RUNS / "Ablation_IaCGOD_TF_runs"])
obj["ratio"] = obj.objective_chars / obj.request_chars
display(obj.groupby("lang").agg(scenarios=("row_number", "size"), median_objectives=("n_objectives", "median"), median_length_ratio=("ratio", "median")).round(2))
print("ImageTagImmutability in rows", obj[obj.ImageTagImmutability].row_number.tolist(), "| MaxConcurrentRuns in rows", obj[obj.MaxConcurrentRuns].row_number.tolist())

,scenarios,median_objectives,median_length_ratio
lang,,,
CFN,50,17.0,3.94
TF,50,17.0,2.84


ImageTagImmutability in rows [172] | MaxConcurrentRuns in rows [183, 240]


## 13. Gemini-3.8-Flash Terraform iterations with the first-attempt overlay

In [14]:
g = load("TF_RealAWS_Gemini38F"); fa = load("TF_RealAWS_Gemini38F_FirstAttempt")
fa_pass = set(fa.loc[(fa.final_validation_passed == True) & (fa.validity == "valid_pass"), "row_number"])
p = g[g["pass"]].assign(iters=lambda d: d.iterations_used.where(~d.row_number.isin(fa_pass), 1))
print(f"passing scenarios {len(p)}; iterations to pass: {p.iters.mean():.2f} (median {p.iters.median():.0f}) with the overlay, "
      f"{p.iterations_used.mean():.2f} (median {p.iterations_used.median():.0f}) without; first-attempt passes {len(fa_pass)}")

passing scenarios 249; iterations to pass: 2.58 (median 2) with the overlay, 3.18 (median 3) without; first-attempt passes 47


## 14. o3-mini on DPIaC-Eval: templates with a critical or high finding

In [15]:
t = pd.read_csv(RES / "Trivy_CFN_DPIaCEval_o3mini_DeployOnly.csv")
ch = (t.severity_critical_failed.fillna(0) + t.severity_high_failed.fillna(0)) > 0
print(f"templates {len(t)}; with a critical or high finding {int(ch.sum())}; free of findings {100 * ((t.total_failed == 0) & t.error.isna()).mean():.1f}%")

templates 153; with a critical or high finding 100; free of findings 13.1%


## 15. Resource recall against strict intent

In [16]:
cv = pd.read_csv(RES / "Coverage_CFN_Ablation_IaCGOD.csv").drop_duplicates("row_number")
m = b[b.lang == "cfn"].merge(cv[["row_number", "coverage_pct"]], on="row_number", how="left")
st = m.intent_strict.fillna(False).astype(bool)
print("CFN templates with full recall that fail strict intent:", int(((m.coverage_pct == 100) & ~st).sum()),
      "| strict with recall below 80%:", int((st & (m.coverage_pct < 80)).sum()))

CFN templates with full recall that fail strict intent: 7 | strict with recall below 80%: 9


## 16. IaC-Eval references that pin a Terraform version

In [17]:
rv = pd.read_csv(RES / "rego_reference_ceiling" / "IaCEval_reference_outputs_relaxed.csv", usecols=["row_number", "required_version_removed"])
print(f"references with a required_version line: {int(rv.required_version_removed.sum())} of {len(rv)}")

references with a required_version line: 227 of 458


## 17. Repair-turn resolution with Wilson intervals (figure coordinates)

In [18]:
haz = pd.read_csv(TABLES / "haz.csv")
haz = haz[haz.at_risk >= 10].copy()
haz["pct"] = (100 * haz.resolved / haz.at_risk).round(1)
lo, hi = wilson(haz.resolved, haz.at_risk)
haz["ci_lo"], haz["ci_hi"] = (100 * lo).round(1), (100 * hi).round(1)
haz["plus"] = (haz.ci_hi - haz.pct).round(1); haz["minus"] = (haz.pct - haz.ci_lo).round(1)
display(haz.drop(columns="hazard"))
for (l, c), d in haz.groupby(["lang", "cls"]):
    print(l, c, " ".join(f"({k},{p:.1f}) += (0,{u:.1f}) -= (0,{v:.1f})" if c == "deploy" else f"({k},{p:.1f})" for k, p, u, v in zip(d.turn_k, d.pct, d.plus, d.minus)))

,lang,cls,turn_k,at_risk,resolved,pct,ci_lo,ci_hi,plus,minus
0,CFN,syntax,1,104,92,88.5,80.9,93.3,4.8,7.6
1,CFN,syntax,2,12,8,66.7,39.1,86.2,19.5,27.6
8,CFN,schema,1,370,269,72.7,67.9,77.0,4.3,4.8
9,CFN,schema,2,101,76,75.2,66.0,82.6,7.4,9.2
10,CFN,schema,3,25,14,56.0,37.1,73.3,17.3,18.9
11,CFN,schema,4,11,5,45.5,21.3,72.0,26.5,24.2
16,CFN,security,1,191,152,79.6,73.3,84.7,5.1,6.3
17,CFN,security,2,39,27,69.2,53.6,81.4,12.2,15.6
18,CFN,security,3,12,9,75.0,46.8,91.1,16.1,28.2
24,CFN,deploy,1,298,114,38.3,32.9,43.9,5.6,5.4


CFN deploy (1,38.3) += (0,5.6) -= (0,5.4) (2,33.7) += (0,7.1) -= (0,6.4) (3,20.5) += (0,8.0) -= (0,6.2) (4,12.4) += (0,8.0) -= (0,5.2) (5,9.4) += (0,8.1) -= (0,4.6) (6,14.3) += (0,9.5) -= (0,6.1) (7,13.6) += (0,10.3) -= (0,6.3) (8,14.0) += (0,11.3) -= (0,6.7)
CFN schema (1,72.7) (2,75.2) (3,56.0) (4,45.5)
CFN security (1,79.6) (2,69.2) (3,75.0)
CFN syntax (1,88.5) (2,66.7)
TF deploy (1,60.7) += (0,4.4) -= (0,4.5) (2,44.7) += (0,7.3) -= (0,7.1) (3,38.4) += (0,9.8) -= (0,9.0) (4,26.2) += (0,12.2) -= (0,9.4) (5,13.3) += (0,12.9) -= (0,7.0) (6,20.5) += (0,15.0) -= (0,9.7) (7,16.1) += (0,16.5) -= (0,9.0) (8,30.8) += (0,19.2) -= (0,14.3)
TF security (1,89.3) (2,57.9)
TF syntax (1,48.8) (2,62.0) (3,58.7) (4,42.3) (5,40.0)


## 18. Passed runs that lost an iteration to the environment

In [19]:
SEC = json.load(open(DATA / "security_checks.json"))
rows = []
for e in [f"{l}_RealAWS_{m}" for l in ("CFN", "TF") for m in ("DSV4F", "GLM53F", "Gemini38F", "Opus55")]:
    l = e.split("_")[0]
    m = pd.read_pickle(CACHE / f"msgs_{e}.pkl")["df"]
    m["iteration"] = m.iteration.astype(int); m = m[m.iteration <= BUDGET]
    m["stage_of"] = [ec.STAGE_OF.get(ec.categorise(l, r.cls, r.stage, r.key, r.sub, r.msg, SEC)[0]) for r in m.itertuples()]
    r = load(e); P = r[r["pass"]]
    env = P.run_id.astype(str).isin(m.loc[m.stage_of == "Environment", "run_id"].astype(str))
    rows.append({"experiment": e, "passed": len(P), "with an environment error": int(env.sum()), "share_pct": round(100 * env.mean(), 1)})
pd.DataFrame(rows)

,experiment,passed,with an environment error,share_pct
0,CFN_RealAWS_DSV4F,239,14,5.9
1,CFN_RealAWS_GLM53F,228,10,4.4
2,CFN_RealAWS_Gemini38F,248,7,2.8
3,CFN_RealAWS_Opus55,246,6,2.4
4,TF_RealAWS_DSV4F,246,51,20.7
5,TF_RealAWS_GLM53F,246,62,25.2
6,TF_RealAWS_Gemini38F,249,82,32.9
7,TF_RealAWS_Opus55,250,22,8.8
